# 11: Leaderboard

**This committed run is 11/12 rows real.** All 10 patterns + both baselines were scored
against real `OPENAI_API_KEY`-backed embeddings/generation/judging, against this repo's
54-chunk pilot corpus -- except pattern 07 (contextual retrieval), which stays mock because
`ANTHROPIC_API_KEY` wasn't set for this run (that pattern's Anthropic-only contextualization
step falls back to `MockLLM` automatically when the key is absent, rather than failing the
whole run). Running this notebook without `OPENAI_API_KEY` set (`RAG_RECIPES_LLM=mock`) scores
all 12 rows under `MockEmbedder`/`MockLLM` instead, which only proves the AGGREGATION PIPELINE
runs end to end -- it says nothing about which pattern actually retrieves or answers better.
`VOYAGE_API_KEY` is NOT needed here, that's `A2_embedding_swap.ipynb`-only.

**This same banner is baked into `outputs/leaderboard.png` itself** (not just this notebook),
since the image can be shared or embedded (e.g. in README.md) independent of this notebook's
surrounding text.

**On "reproducible": retrieval scores are, cost figures aren't bit-for-bit.** `hit@k`/`mrr` are fully deterministic given the same corpus/qa_set/embedder --
confirmed identical across repeated runs. `usd_per_query`/`eval_usd` are NOT bit-for-bit identical
across runs, though: `evals/judges.py` caches judge calls to disk (`outputs/.judge_cache/`,
gitignored) keyed on (judge_type, model, prompt template, question, context, answer) specifically
to avoid re-billing identical judge calls on reruns -- this is working as designed,
not a bug, but it means the printed cost reflects MARGINAL cost given your local cache state, not a
from-scratch total. For a clean-cache cost baseline, delete `outputs/.judge_cache/` before running.

## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

import time
from pathlib import Path

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes import AnswerWithCitations
from recipes.embeddings import get_embedder
from recipes.llm import MockLLM, get_anthropic_llm, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()
embedder = get_embedder()
# get_anthropic_llm() only falls back to mock when RAG_RECIPES_LLM=="mock"
# globally, not when ANTHROPIC_API_KEY specifically is absent -- gate on the
# actual key so a missing Anthropic key degrades pattern 07 to mock instead of
# hard-crashing the whole leaderboard run partway through (same fix as
# A2_embedding_swap.ipynb's voyage-4 row).
if os.environ.get("ANTHROPIC_API_KEY"):
    anthropic_llm = get_anthropic_llm()
else:
    print("ANTHROPIC_API_KEY not set -- pattern 07 will run under MockLLM")
    anthropic_llm = MockLLM()

if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

GENERATION_MODEL = "gpt-4.1-mini-2025-04-14"
PROMPT_TEMPLATE = Path("../prompts/generation_prompt.txt").read_text(encoding="utf-8")


ANTHROPIC_API_KEY not set -- pattern 07 will run under MockLLM


## Baseline builders (00 / 00b -- reproduced inline, never exported to recipes/, see those notebooks)

In [3]:
def make_no_rag_pattern(llm):
    def retrieve_and_answer(question: str, k: int = 5) -> AnswerWithCitations:
        start = time.perf_counter()
        prompt = PROMPT_TEMPLATE.format(context="", question=question)
        response = llm.complete(prompt=prompt, model=GENERATION_MODEL, temperature=0.0)
        latency_ms = (time.perf_counter() - start) * 1000
        return AnswerWithCitations(
            answer=response.text, retrieved_chunk_ids=[], latency_ms=latency_ms,
            input_tokens=response.input_tokens, output_tokens=response.output_tokens,
            cached_input_tokens=response.cached_input_tokens,
        )
    return retrieve_and_answer


def make_long_context_pattern(corpus_by_id, llm):
    all_chunk_ids = list(corpus_by_id.keys())
    full_context = "\n\n".join(f"[{cid}] " + corpus_by_id[cid]["text"] for cid in all_chunk_ids)

    def retrieve_and_answer(question: str, k: int = 5) -> AnswerWithCitations:
        start = time.perf_counter()
        prompt = PROMPT_TEMPLATE.format(context=full_context, question=question)
        response = llm.complete(prompt=prompt, model=LONG_CONTEXT_MODEL, temperature=0.0)
        latency_ms = (time.perf_counter() - start) * 1000
        return AnswerWithCitations(
            answer=response.text, retrieved_chunk_ids=all_chunk_ids, latency_ms=latency_ms,
            input_tokens=response.input_tokens, output_tokens=response.output_tokens,
            cached_input_tokens=response.cached_input_tokens,
        )
    return retrieve_and_answer

LONG_CONTEXT_MODEL = "gpt-5.4-mini-2026-03-17"


## Pattern builders (reuse recipes/*.py directly, no logic redefined)

In [4]:
from recipes.agentic import make_retrieve_and_answer as build_10
from recipes.bm25 import make_retrieve_and_answer as build_02
from recipes.contextual import make_retrieve_and_answer as build_07
from recipes.hybrid import make_retrieve_and_answer as build_03
from recipes.hyde import make_retrieve_and_answer as build_05
from recipes.multi_hop import make_retrieve_and_answer as build_09
from recipes.multi_query import make_retrieve_and_answer as build_06
from recipes.naive_dense import make_retrieve_and_answer as build_01
from recipes.rerank import make_retrieve_and_answer as build_04
from recipes.self_query import make_retrieve_and_answer as build_08

# Pattern 07's one-time contextualization cost log -- not part of the
# leaderboard's per-query cost accounting (see recipes/contextual.py).
cost_log_07: list[dict] = []

# name, type, factory -- ordered 00/00b then 01-10 for readability while
# running; the results table below sorts by hit@10 for display.
PATTERNS = [
    ("00_baseline_no_rag", "baseline", lambda: make_no_rag_pattern(llm)),
    ("00b_long_context_baseline", "baseline", lambda: make_long_context_pattern(corpus_by_id, llm)),
    ("01_naive_dense", "pattern", lambda: build_01(corpus_by_id, embedder=embedder, llm=llm)),
    ("02_bm25", "pattern", lambda: build_02(corpus_by_id, llm=llm)),
    ("03_hybrid_rrf", "pattern", lambda: build_03(corpus_by_id, embedder=embedder, llm=llm)),
    ("04_rerank", "pattern", lambda: build_04(corpus_by_id, llm=llm)),
    ("05_hyde", "pattern", lambda: build_05(corpus_by_id, embedder=embedder, llm=llm)),
    ("06_multi_query", "pattern", lambda: build_06(corpus_by_id, embedder=embedder, llm=llm)),
    ("07_contextual", "pattern", lambda: build_07(corpus_by_id, embedder=embedder, llm=llm, anthropic_llm=anthropic_llm, cost_log=cost_log_07)),
    ("08_self_query", "pattern", lambda: build_08(corpus_by_id, embedder=embedder, llm=llm)),
    ("09_multi_hop", "pattern", lambda: build_09(corpus_by_id, embedder=embedder, llm=llm)),
    ("10_agentic", "pattern", lambda: build_10(corpus_by_id, embedder=embedder, llm=llm)),
]


## Run loop

In [5]:
results = []
row_types = {}
for name, row_type, factory in PATTERNS:
    print(f"Running {name}...")
    recipe_fn = factory()
    trace_path = "../outputs/agentic_traces.jsonl" if name == "10_agentic" else None
    # 00b actually generates with gpt-5.4-mini, not run_pattern()'s
    # gpt-4.1-mini default -- must be passed explicitly or cost_usd() would
    # silently price its real tokens at the wrong (cheaper) rate. Same
    # override applied directly to 00b_long_context_baseline.ipynb's own
    # run_pattern() call.
    gen_model_override = {"generation_model": LONG_CONTEXT_MODEL} if name == "00b_long_context_baseline" else {}
    result = run_pattern(
        recipe_fn=recipe_fn, qa_set=qa_set, corpus_by_id=corpus_by_id, llm=judge_llm,
        pattern_name=name, judges_enabled=True, verbose=False, trace_output_path=trace_path,
        **gen_model_override,
    )
    # Sanity check: under mock, nothing should ever fail -- a nonzero count
    # here is a real bug (e.g. a pattern module regression), not expected
    # noise, and must not be silently swallowed into "just a low score."
    assert result.n_errors == 0, f"{name}: {result.n_errors} recipe_fn failures under mock"
    assert result.n_judge_errors == 0, f"{name}: {result.n_judge_errors} judge failures under mock"
    assert result.n_accounting_errors == 0, f"{name}: {result.n_accounting_errors} accounting failures under mock"
    results.append(result)
    row_types[name] = row_type

print(f"\nDone. {len(results)} patterns scored.")


Running 00_baseline_no_rag...


Running 00b_long_context_baseline...


Running 01_naive_dense...


Running 02_bm25...


Running 03_hybrid_rrf...


...

Running 07_contextual...


Running 08_self_query...


Running 09_multi_hop...


Running 10_agentic...



Done. 12 patterns scored.


## Table assembly

In [6]:
def ci_str(ci):
    if ci is None:
        return "--"
    return f"{ci.mean:.3f} [{ci.lower:.3f}, {ci.upper:.3f}]"

ranked = sorted(results, key=lambda r: r.hit_at_10.mean, reverse=True)

full_rows = []
for rank, r in enumerate(ranked, start=1):
    full_rows.append({
        "rank": rank,
        "pattern": r.pattern_name,
        "type": row_types[r.pattern_name],
        "hit@3": ci_str(r.hit_at_3),
        "hit@10": ci_str(r.hit_at_10),
        "mrr": ci_str(r.mrr),
        "faithfulness": ci_str(r.faithfulness),
        "answer_relevance": ci_str(r.answer_relevance),
        "citation_accuracy": ci_str(r.citation_accuracy),
        "filter_accuracy": ci_str(r.filter_accuracy),
        "p50_latency_ms": f"{r.p50_latency_ms:.1f}",
        "p95_latency_ms": f"{r.p95_latency_ms:.1f}",
        "usd_per_query": f"${r.usd_per_query:.5f}",
        "eval_usd": f"${r.eval_usd:.4f}",
    })

total_eval_usd = sum(r.eval_usd for r in results)
print(f"Total eval_usd across all 12 runs: ${total_eval_usd:.4f}")


Total eval_usd across all 12 runs: $1.8895


## Write outputs/leaderboard.md

In [7]:
md_lines = [
    "# rag-recipes leaderboard",
    "",
    ("**11/12 rows real (OpenAI); pattern 07 mock, needs ANTHROPIC_API_KEY** -- see "
     "notebooks/11_leaderboard.ipynb for the full disclaimer."),
    "",
    f"- Generated: {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}",
    f"- Total eval_usd: ${total_eval_usd:.4f}",
    "",
    ("| rank | pattern | type | hit@3 | hit@10 | mrr | faithfulness | answer_relevance | "
     "citation_accuracy | filter_accuracy | p50_ms | p95_ms | usd/query | eval_usd |"),
    "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|",
]
for row in full_rows:
    md_lines.append(
        f"| {row['rank']} | {row['pattern']} | {row['type']} | {row['hit@3']} | {row['hit@10']} | "
        f"{row['mrr']} | {row['faithfulness']} | {row['answer_relevance']} | "
        f"{row['citation_accuracy']} | {row['filter_accuracy']} | {row['p50_latency_ms']} | "
        f"{row['p95_latency_ms']} | {row['usd_per_query']} | {row['eval_usd']} |"
    )

Path("../outputs/leaderboard.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")
print("wrote ../outputs/leaderboard.md")


wrote ../outputs/leaderboard.md


## Write outputs/leaderboard.png

In [8]:
import matplotlib.pyplot as plt

condensed_cols = ["rank", "pattern", "type", "hit@10", "faithfulness", "usd_per_query"]
cell_text = [[str(row[c]) for c in condensed_cols] for row in full_rows]

fig, ax = plt.subplots(figsize=(10, 0.4 * len(full_rows) + 1.5))
ax.axis("off")
ax.set_title(
    "rag-recipes leaderboard  --  11/12 rows real (OpenAI); pattern 07 mock (see notebooks/11_leaderboard.ipynb)",
    fontsize=8.5, color="darkgreen", wrap=True,
)
col_widths = [0.06, 0.24, 0.12, 0.19, 0.19, 0.16]  # wider "pattern" column -- longest name (00b_long_context_baseline) was clipped at the default even width
table = ax.table(
    cellText=cell_text, colLabels=condensed_cols, loc="center", cellLoc="left", colWidths=col_widths
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4)

fig.tight_layout()
fig.savefig("../outputs/leaderboard.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("wrote ../outputs/leaderboard.png")


wrote ../outputs/leaderboard.png
